In [1]:
import pandas as pd
import os

In [2]:
DATA_DIR = os.path.join("..", "data")  # same relative-path pattern as before

train = pd.read_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))
test = pd.read_parquet(os.path.join(DATA_DIR, "test_ratings.parquet"))

print(train.shape, test.shape)

(118387327, 4) (29596832, 4)


In [ ]:
# Count positive interactions per anime, using train only
popularity = train[train['is_positive'] == 1].groupby('anime_id').size().sort_values(ascending=False)

#K most popular shows
K = 10
top_k_popular = popularity.head(K).index.tolist()

print(top_k_popular)

#Find the precision and recall rate
def precision_recall_at_k(test_df, recommended_items, k):
    total_relevant = 0
    total_recommended_relevant = 0
    
    for user_id, group in test_df[test_df['is_positive'] == 1].groupby('user_id'):
        actual_positive = set(group['anime_id'])
        recommended = set(recommended_items[:k])
        
        hits_this_user = len(actual_positive & recommended)
        total_recommended_relevant += hits_this_user
        total_relevant += len(actual_positive)
    
    # Recall: of everything the user actually liked, what fraction did we recommend?
    recall = total_recommended_relevant / total_relevant if total_relevant > 0 else 0
    
    # Precision: of everything we recommended, what fraction did users actually like?
    num_users = test_df['user_id'].nunique()
    precision = total_recommended_relevant / (num_users * k)
    
    return precision, recall

precision, recall = precision_recall_at_k(test, top_k_popular, K)
print(f"Recall@{K}: {recall:.4f}")
print(f"Precision@{K}: {precision:.4f}")


[20, 2, 92, 2376, 99, 726, 100, 1147, 1160, 1167, 1154, 2977, 2384, 3034, 1186, 6, 3035, 3290, 1091, 71, 1109, 36, 218, 1138, 59]
Precision@25: 0.0541
Recall@25: 0.1342
